In [ ]:
import os
import shutil
from google.colab import drive
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# 1. Монтируем диск
drive.mount('/content/drive', force_remount=True)

# 2. Автоматический поиск папки датасета на Google Диске (независимо от точного имени родительской папки)
drive_root = '/content/drive/MyDrive'
target_folder = None

print("Ищем папку LEVIR_processed_5K на Google Диске...")
for root, dirs, files in os.walk(drive_root):
    if 'LEVIR_processed_5K' in dirs:
        target_folder = os.path.join(root, 'LEVIR_processed_5K')
        break

if target_folder:
    print(f"Папка найдена: {target_folder}")

    # 3. Копируем датасет в локальное хранилище Colab (/content/) для максимальной скорости
    local_dir = '/content/LEVIR_processed_5K'
    if not os.path.exists(local_dir):
        print("Копирование датасета на локальный диск Colab...")
        shutil.copytree(target_folder, local_dir)
        print("Копирование успешно завершено!")
    else:
        print("Датасет уже присутствует в локальной памяти.")

    DATA_DIR = local_dir
else:
    raise FileNotFoundError(
        "Папка 'LEVIR_processed_5K' не найдена на Google Диске. "
        "Убедитесь, что она загружена в ваш MyDrive."
    )

# 1. Загрузчик подготовленных данных (LEVIR-CD 256x256)
class LevirCDDataset(Dataset):
    def __init__(self, data_dir, split='train'):
        self.img_A_dir = os.path.join(data_dir, split, 'A')
        self.img_B_dir = os.path.join(data_dir, split, 'B')
        self.label_dir = os.path.join(data_dir, split, 'label')
        self.filenames = sorted(os.listdir(self.img_A_dir))

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img_a = cv2.imread(os.path.join(self.img_A_dir, fname))
        img_b = cv2.imread(os.path.join(self.img_B_dir, fname))
        mask = cv2.imread(os.path.join(self.label_dir, fname), cv2.IMREAD_GRAYSCALE)

        # Перевод в RGB и нормализация в диапазон [0, 1]
        img_a = cv2.cvtColor(img_a, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        img_b = cv2.cvtColor(img_b, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

        # Пороговая бинаризация целевой маски
        mask = (mask > 127).astype(np.float32)

        # Преобразование в тензоры PyTorch (Channel, Height, Width)
        img_a = torch.tensor(img_a).permute(2, 0, 1)
        img_b = torch.tensor(img_b).permute(2, 0, 1)
        mask = torch.tensor(mask).unsqueeze(0)

        return img_a, img_b, mask


# 2. Реализация базовой архитектуры Siamese U-Net
class SiameseUNet(nn.Module):
    def __init__(self):
        super(SiameseUNet, self).__init__()
        # Общая ветвь извлечения признаков
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )
        # Декодер с транспонированной сверткой
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x_a, x_b):
        feat_a = self.encoder(x_a)
        feat_b = self.encoder(x_b)
        # Объединение карт признаков T1 и T2 по каналам
        concat_features = torch.cat([feat_a, feat_b], dim=1)
        output_mask = self.decoder(concat_features)
        return output_mask


# 3. Исполняемый блок запуска первого рабочего эксперимента
if __name__ == "__main__":
    BATCH_SIZE = 8
    EPOCHS = 5
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Инициализация датасетов и DataLoader
    train_dataset = LevirCDDataset(DATA_DIR, split='train')
    val_dataset = LevirCDDataset(DATA_DIR, split='val')

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Инициализация модели, loss-функции и оптимизатора
    model = SiameseUNet().to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)

    print(f"[INFO] Запуск первого эксперимента на устройстве: {DEVICE}")

    # Обучение базовой модели
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for img_a, img_b, masks in tqdm(train_loader, desc=f"Эпоха {epoch + 1}/{EPOCHS}"):
            img_a, img_b, masks = img_a.to(DEVICE), img_b.to(DEVICE), masks.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(img_a, img_b)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_loss = train_loss / len(train_loader)
        print(f"Эпоха {epoch + 1}/{EPOCHS} | Train Loss: {avg_loss:.4f}")

output_model_path = '/content/drive/MyDrive/siamese_unet_levir.pth'
torch.save(model.state_dict(), output_model_path)
print(f"[INFO] Модель сохранена на Google Диск: {output_model_path}")

Mounted at /content/drive
Ищем папку LEVIR_processed_5K на Google Диске...
Папка найдена: /content/drive/MyDrive/Компьютерное зрение (Кейс LEVIR-CD)/LEVIR_processed_5K
Копирование датасета на локальный диск Colab (это займет пару секунд)...
Копирование успешно завершено!
[INFO] Запуск первого эксперимента на устройстве: cuda


Эпоха 1/5: 100%|██████████| 393/393 [01:15<00:00,  5.20it/s]


Эпоха 1/5 | Train Loss: 0.3258


Эпоха 2/5: 100%|██████████| 393/393 [01:13<00:00,  5.36it/s]


Эпоха 2/5 | Train Loss: 0.2562


Эпоха 3/5: 100%|██████████| 393/393 [01:14<00:00,  5.30it/s]


Эпоха 3/5 | Train Loss: 0.2402


Эпоха 4/5: 100%|██████████| 393/393 [01:15<00:00,  5.21it/s]


Эпоха 4/5 | Train Loss: 0.2258


Эпоха 5/5: 100%|██████████| 393/393 [01:16<00:00,  5.14it/s]

Эпоха 5/5 | Train Loss: 0.2133
